In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import seaborn as sns
pio.renderers.default='colab'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load in ind and team data and Combine. Filter to only transfers

In [ ]:
team_df = pd.read_csv('/content/drive/MyDrive/Basketball_analysis/team_stats.csv')
df = pd.read_csv('/content/drive/MyDrive/Basketball_analysis/season_stats.csv')

In [ ]:
combined_df = df.merge(
    team_df,
    on=["team", "year","conf"],
    how="left"
)


In [ ]:
team_counts = (
    df.groupby("id")["team"]
      .nunique()
      .reset_index(name="num_teams")
)
print(team_counts)

            id  num_teams
0        37479          1
1        39014          1
2        39412          1
3        39780          1
4        40204          2
...        ...        ...
11525   133656          1
11526   133659          1
11527   133660          1
11528   133661          1
11529  2450496          1

[11530 rows x 2 columns]


In [ ]:
combined_df = combined_df[(combined_df['g'] >= 5)]

In [ ]:
combined_df = combined_df.drop(columns=["num","g","ppg","oreb","dreb","rpg","apg","tov","spg","bpg",
                  "efg","id","fgm","ftm","fta","two_a","two_m","three_m","three_a",
                  "dunk_m","dunk_a","dunk_pct","rim_m","rim_a","mid_m","mid_a","pick","fga",
                  "barthag_rk","adj_o_rk","adj_d_rk","adj_t_rk","seed","rec"], axis=1)

In [ ]:
transfers = team_counts[team_counts["num_teams"] > 1]
transfers.shape

(3466, 2)

In [ ]:
combined_df = combined_df.sort_values(["player", "year"])

combined_df["previous_team"] = (
    combined_df.groupby("player")["team"]
      .shift(1)
)

combined_df["previous_conference"] = (
    combined_df.groupby("player")["conf"]
      .shift(1)
)


combined_df["is_transfer"] = (
    combined_df["previous_team"].notna() &
    (combined_df["team"] != combined_df["previous_team"])
)

combined_df.loc[~combined_df["is_transfer"], "previous_team"] = pd.NA
combined_df.loc[~combined_df["is_transfer"], "previous_conference"] = pd.NA

combined_df["previous_ortg"] = (
    combined_df.groupby("player")["ortg"]
      .shift(1)
)

combined_df["previous_drtg"] = (
    combined_df.groupby("player")["drtg"]
      .shift(1)
)

combined_df["previous_ts"] = (
    combined_df.groupby("player")["ts"]
      .shift(1)
)

combined_df["previous_usg"] = (
    combined_df.groupby("player")["usg"]
      .shift(1)
)

combined_df["previous_porpag"] = (
    combined_df.groupby("player")["porpag"]
      .shift(1)
)

combined_df["previous_adj_oe"] = (
    combined_df.groupby("player")["adj_oe"]
      .shift(1)
)

combined_df["previous_adj_de"] = (
    combined_df.groupby("player")["adj_de"]
      .shift(1)
)

combined_df["previous_bpm"] = (
    combined_df.groupby("player")["bpm"]
      .shift(1)
)

combined_df["previous_team_adjo"] = (
    combined_df.groupby("player")["adj_o"]
      .shift(1)
)

combined_df["previous_team_adjd"] = (
    combined_df.groupby("player")["adj_d"]
      .shift(1)
)

combined_df["previous_barthag"] = (
    combined_df.groupby("player")["barthag"]
      .shift(1)
)
combined_df["previous_oreb_rate"] = (
    combined_df.groupby("player")["oreb_rate"]
      .shift(1)
)

In [ ]:
combined_df["change_ortg"] = combined_df["ortg"] - combined_df["previous_ortg"]
combined_df["change_drtg"] = combined_df["drtg"] - combined_df["previous_drtg"]
combined_df["change_ts"] = combined_df["ts"] - combined_df["previous_ts"]
combined_df["change_usg"] = combined_df["usg"] - combined_df["previous_usg"]
combined_df["change_porpag"] = combined_df["porpag"] - combined_df["previous_porpag"]
combined_df["change_adj_oe"] = combined_df["adj_oe"] - combined_df["previous_adj_oe"]
combined_df["change_adj_de"] = combined_df["adj_de"] - combined_df["previous_adj_de"]
combined_df["change_bpm"] = combined_df["bpm"] - combined_df["previous_bpm"]
combined_df["change_team_adjo"] = combined_df["adj_o"] - combined_df["previous_team_adjo"]
combined_df["change_team_adjd"] = combined_df["adj_d"] - combined_df["previous_team_adjd"]
combined_df["change_barthag"] = combined_df["barthag"] - combined_df["previous_barthag"]
combined_df["change_oreb_rate"] = combined_df["oreb_rate"] - combined_df["previous_oreb_rate"]

In [ ]:
df_transfers = combined_df[(combined_df['is_transfer']==True)]
df_transfers.shape

(3982, 108)

In [ ]:
df_transfers.columns

Index(['player', 'pos', 'exp', 'num', 'hgt', 'team', 'conf', 'g', 'min', 'mpg',
       ...
       'change_ts', 'change_usg', 'change_porpag', 'change_adj_oe',
       'change_adj_de', 'change_bpm', 'change_team_adjo', 'change_team_adjd',
       'change_barthag', 'change_oreb_rate'],
      dtype='object', length=108)

In [ ]:
df_transfers.to_csv(
    "/content/drive/MyDrive/Basketball_analysis/transfers.csv",
    index=False
)

SJU Stats

Scoring Offense: 73.1 PPG (150)

Scoring Defense: 70.2 (71)

Ast/Tov: 1.32 (111)

Blocks Per Game: 4.7 (26)

Rebound Margin: 2.9 (104)




In [ ]:
correlations = df_transfers.corr(numeric_only=True)['change_oreb_rate'].sort_values(ascending=False)
print(correlations)

change_oreb_rate      1.000000
pick                  0.369400
oreb_rate             0.351005
change_adj_oe         0.235821
oreb                  0.188621
                        ...   
previous_ortg        -0.104786
previous_usg         -0.142932
previous_adj_oe      -0.146139
previous_oreb_rate   -0.469794
is_transfer                NaN
Name: change_oreb_rate, Length: 100, dtype: float64


In [ ]:
top_oreb_rate_change = df_transfers.nlargest(100, 'change_oreb_rate')

selected_columns = [
    'oreb_rate', 'pos', 'hgt', 'team', 'mpg', 'barthag', 'previous_barthag'
]

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print(top_oreb_rate_change[selected_columns])

# Reset display options to default after showing the full dataset (optional)
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')

       oreb_rate        pos     hgt                    team      mpg  \
3572        38.3          C     6-9              St. Thomas   4.6364   
7131        30.1       PF/C     6-9                Ohio St.   2.8571   
5519        22.8  Stretch 4     6-3           Southern Utah   4.3000   
11117       20.1       PF/C     6-7                  Temple   5.2727   
2853        31.6          C    6-10           Virginia Tech   2.5000   
11076       22.1          C    6-10              Fresno St.   4.8571   
6505        14.5       PF/C     6-7              High Point  20.5161   
17484       18.1       PF/C   5-Jun              Texas Tech   1.4444   
16240       13.5  Stretch 4  11-May              Charleston   1.6714   
14273       13.4  Stretch 4     6-6             The Citadel  14.1111   
20061       14.7          C   9-Jun          Incarnate Word  16.5778   
5635        17.3  Stretch 4     6-3                 Arizona   1.7500   
16129       14.0     Wing F   5-Jun                 Alabama   7.

In [ ]:
print(combined_df['oreb_rate'].mean)

<bound method Series.mean of 7558     4.8
9221     3.9
13007    2.2
20806    1.6
438      2.6
        ... 
6244     1.4
10612    2.2
10010    3.7
13685    3.2
17486    1.5
Name: oreb_rate, Length: 22497, dtype: float64>


In [ ]:
from sklearn.model_selection import train_test_split
X = df_transfers[[
    # Player profile
    'pos', 'exp', 'hgt',

    # Team/context
    'team', 'conf', 'barthag', 'adj_t', 'wab',
    'adj_de', 'drtg', 'stops', 'dbpm',

    # Playing time
    'min', 'mpg',

    # Strength of schedule
    'nc_elite_sos', 'nc_fut_sos', 'nc_cur_sos',
    'ov_elite_sos', 'ov_fut_sos', 'ov_cur_sos',

    # Transfer/previous season data
    'previous_ortg', 'previous_drtg', 'previous_ts', 'previous_usg',
    'previous_porpag', 'previous_adj_oe', 'previous_adj_de', 'previous_bpm',
    'previous_team_adjo', 'previous_team_adjd', 'previous_barthag',

    # Defensive/rebounding (less correlated with offensive targets)
    'dreb_rate', 'blk', 'stl', 'pfr',
]]
y = df_transfers[['adj_oe', 'bpm', 'usg', 'porpag', 'ast_to', 'oreb_rate', 'three_pct']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing, 80% for training
    random_state=42
)

In [ ]:
X_train.to_csv(
    "/content/drive/MyDrive/Basketball_analysis/X_train.csv",
    index=False
)
y_train.to_csv(
    "/content/drive/MyDrive/Basketball_analysis/y_train.csv",
    index=False
)
X_test.to_csv(
    "/content/drive/MyDrive/Basketball_analysis/X_test.csv",
    index=False
)
y_test.to_csv(
    "/content/drive/MyDrive/Basketball_analysis/y_test.csv",
    index=False
)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Identify numerical and categorical features
categorical_features = ['pos', 'exp', 'hgt', 'team', 'conf']
numerical_features = X_train.columns.difference(categorical_features)

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_processed, y_train['adj_oe'])

RandomForestRegressor(random_state=42)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

preds = model.predict(X_test_processed)
print(f"MAE: {mean_absolute_error(y_test['adj_oe'], preds):.2f}")
print(f"R²:  {r2_score(y_test['adj_oe'], preds):.3f}")

MAE: 9.59
R²:  0.396


In [ ]:
from sklearn.impute import SimpleImputer

imputer_y = SimpleImputer(strategy='mean')
y_train_bpm_imputed = imputer_y.fit_transform(y_train[['bpm']])

model.fit(X_train_processed, y_train_bpm_imputed.ravel())

ValueError: Input y contains NaN.